In [ ]:
# Import Libraries and modules

# libraries that are used for analysis and visualization

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Impoting data preprocessing libraries
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
# Importing model selection libraries.
from sklearn.model_selection import train_test_split


In [ ]:
# Importing metrics for model evaluation.
from sklearn.metrics import confusion_matrix, accuracy_score

In [ ]:
# Importing machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

load the CARDIOVASCULAR RISK DATA

In [ ]:
#Dataset First View
risk_df=pd.read_csv('cardiovascular_risk_data.csv',na_values=["NA", "N/A", "null", "None", "?", "-"])

# Viewing the top 5 rows to take a glimpse of the data
risk_df.head(5)

In [ ]:
# Viewing the last 5 rows
risk_df.tail(5)

In [ ]:
# Dataset Rows & Columns 
risk_df.shape

In [ ]:
print(f'number of rows : {risk_df.shape[0]}  \nnumber of columns : {risk_df.shape[1]}')

### Dataset Information

In [ ]:
risk_df.info()

## Duplicate Values

### How important is it to get rid of duplicate records in my data?

The mere presence of repeated data in the dataset is referred to as "duplication." This could be caused by incorrect data entry or procedures for collecting data. We can save time and money by not sending the same data to the machine learning model multiple times by removing duplicate data from our set.

In [ ]:
# Checking Duplicate Values
risk_df.duplicated().sum()

Insight: We found that there were no duplicate entries in the above data.

In [ ]:
'''drop id column'''
risk_df=risk_df.drop(['id'],axis=1,errors='ignore')

## Missing Values/Null Values

### Why dealing with missing values is necessary?
There are frequently a lot of missing values in the actual data. Corrupted or missing data may result in missing values. Since many machine-learning algorithms do not support missing values, missing data must be handled during the dataset's pre-processing. Therefore, we begin by looking for values that are missing.

In [ ]:
# Missing Values/Null Values Count
risk_df.isnull().sum()

Insight: We can see there are null values present in totChol, BMI, glucose, BPMeds, cigsPerDay, education has some null values and heartRate has 1 null value.

In [ ]:
# Missing Values Percentage
round(risk_df.isnull().sum()/len(risk_df)*100,2)

### Insight: glucose column has significant number of missing values

In [ ]:
# Visualizing the missing values
import missingno as msno
msno.bar(risk_df,color='orange',sort='ascending',fontsize=24)

In [ ]:
# Visualizing the missing values using Heatmap
plt.figure(figsize=(12,4))
sns.heatmap(risk_df.isna())

### Insight: There is no relation between any columns for missingness
if there would be same pattern for white lines in heatmap then it means two columns missingness have same relation, also if white lines are clustered in some part of a column then we say it is not MCAR but MAR or MNAR and data may be time-ordered or rows represents specific subgroups or dataset made by merging data

## What did you know about your dataset?

The data comes from an ongoing cardiovascular study of Framingham, Massachusetts, residents. The purpose of the classification is to determine whether the patient is at risk for coronary heart disease (CHD) in the ten years to come. The data about the patients is provided by the dataset. It incorporates more than 4,000 records and 15 ascribes.
 
 A classification algorithm is a method of supervised learning that divides data into various classes by utilizing data training. Data or observations are used to train classification predictive modeling, and new observations are categorized into classes or groups.
 
 There are 3390 rows and 16 columns in the dataset. In the 'education', 'cigsPerDay', 'BPMeds', 'totChol', 'BMI', 'heartRate', and 'glucose', there are missing values. The dataset does not contain any duplicate values.
 
 A potential risk factor exists for each attribute. Demographic, behavioral, and medical risk factors are these characteristics.

 First, we should try to understand the dataset through EDA, and then we can deal with null values later.

In [ ]:
#Understanding Your Variables
risk_df.columns

###  Statistical Summary

In [ ]:
# Dataset Describe    (used to get statistics of numerical columns)
risk_df.describe().T

## How to analyze
### diaBP feature
mean=82.8, median=82 approx equal hence not skewed, the distribution is usually symmetric

Compare Quartile Distances
median-Q1=82-74=8
Q3-median=90-82=8 
hence distribution is very symmetric.

Detecting Outliers (Using IQR Rule)
IQR=Q3−Q1=90-74=16
Q1−1.5×IQR=74-1.5(16)=50
Q3+1.5×IQR=90+1.5(16)=114

min=48, max=142 hence outliers exist

Most values fall between mean-std , mean+std

 As can be seen in the statistical summary, there is skewness and outliers present in the continuous features 'cigsperday', 'totchol', 'sysbp', 'diebp', 'BMI', 'heartrate', and 'glucose' because there is such a large difference between the 75% percentile value and the maximum value.

In [ ]:
#  Variables Description

**Demographic**

*age  : Age of the patient (Continuous - Although the recorded ages have been truncated to whole numbers, the concept of age is continuous)

education : level of education from 1 to 4 (Ordinal Variable)

sex : male or female ("M" or "F")*




**Behavioral**

*is_smoking : whether or not the patient is a current smoker ("YES" or "NO")

cigsPerDay : the number of cigarettes that the person smoked on average in one day (can be considered continuous as one can have any number of cigarettes, even half a cigarette.)
*



**Medical( history)**

*BPMeds : whether or not the patient was on blood pressure medication (Nominal)

prevalentStroke : whether or not the patient had previously had a stroke (Nominal)

prevalentHyp :whether or not the patient was hypertensive (Nominal)

diabetes : whether or not the patient had diabetes (Nominal)*



**Medical(current)**

*totChol :total cholesterol level (Continuous)

sysBP : systolic blood pressure (Continuous)

diaBP : diastolic blood pressure (Continuous)

BMI : Body Mass Index (Continuous)

heartRate : heart rate (Continuous - In medical research, variables such as heart rate though in fact 
discrete, yet are considered continuous because of large number of possible values.)

glucose : glucose level (Continuous)*



**Predict variable (desired target)**

TenYearCHD :(binary: “1”, means “Yes”, “0” means “No”)**

In [ ]:
#Check Unique Values for each variable.
# this helps in Detect Data Errors, Remove useless columns, Decide encoding strategy

for i in risk_df.columns:
  print("No. of unique values in",i,"is",risk_df[i].nunique())

## Numeric and Categorical features

In [ ]:
numeric_features = []
categorical_features = []

''' splitting features into numeric and categoric. '''

'''
If feature has more than 10 categories we will consider it
as numerical_features, remaining features will be added to categorical_features.
'''

for col in risk_df.columns:
    if risk_df[col].nunique()>10:
        numeric_features.append(col)
    else:
        categorical_features.append(col)

print(f'numeric feature : {numeric_features}')
print(f'category feature : {categorical_features}')

In [ ]:
'''Data Distribution of Numeric features'''

plt.figure(figsize=(12,4))

plt.suptitle('Data Distibution of Numeric Features', fontsize=20, fontweight='bold', y=1.02)

for i,col in enumerate(numeric_features):

  plt.subplot(2, 4, i+1)                       # subplots 2 rows, 4 columns

  # kde plots
  sns.histplot(risk_df[col])  
  # x-axis label
  plt.xlabel(col)
plt.tight_layout()

Insight from histogram=> cigsPerDay: Many values are 0, meaning many participants do not smoke,smaller groups smoke 20 cigarettes per day,The histogram is highly right-skewed.

In [ ]:
plt.figure(figsize=(12,4))

plt.suptitle('Data Distibution of Numeric Features', fontsize=20, fontweight='bold', y=1.02)

for i,col in enumerate(numeric_features):

  plt.subplot(2, 4, i+1)                       # subplots 2 rows, 4 columns

  # kde plots
  sns.boxplot(risk_df[col])  
  # x-axis label
  plt.xlabel(col)
plt.tight_layout()

### Analysis

The boxplot of BMI,sysBP shows several outliers above the upper whisker.

The boxplot of diaBP,heartRate,glucose shows several outliers above the upper whisker and many below the lower whisker.

cigsPerDay shows few outliers above the upper whisker.

glucose,cigsPerDay are highly right skewed as for cigsPerDay 75 percent people cigsPerDay count is less than 20 and for glucose 75 percent people glucose value is less than 87.

sysBP,totChol are right skewed.

In [ ]:
# figsize
plt.figure(figsize=(15,5))
# title
plt.suptitle('Univariate Analysis of Categorical Features', fontsize=20, fontweight='bold', y=1.02)

for i,col in enumerate(categorical_features):
  plt.subplot(2, 4, i+1)            # subplot of 2 rows and 4 columns

  # countplot
  sns.countplot(x=risk_df[col])
  # x-axis label
  plt.xlabel(col)
plt.tight_layout()

<span style="color:orange"> The countplot for TenYearCHD is severely skewed. Roughly 85% of participants do not develop CHD hence model might achieve high "accuracy" simply by predicting everyone is healthy</span>

<span style="color:orange">Observed significant class imbalance in binary predictors (BPMeds, prevalentStroke, Diabetes)</span>

### plotting graph to analyze age with respect to heartrate which are having Disease or No Disease

In [ ]:
# figsize
plt.figure(figsize=(10,5))
# scatterplot
sns.scatterplot(x=risk_df['age'], y=risk_df['heartRate'], hue=risk_df['TenYearCHD'])
# title
plt.title('Heart Disease wrt Age and heartRate')
plt.legend(['Disease', 'No Disease'])

### Insight: younger ages.	It is rare to see CHD cases in participants under 35 in this dataset, even with high heart rates.

In [ ]:
'''No benefit of this regplot as y axis reperesent yes or no and this plot is highly 
effected by outliers so no real meaning can be drawn'''

# # Checking Linearity of all numerical features with our target variable

# # figsize
# plt.figure(figsize=(15,5))
# # title
# plt.suptitle('Bivariate Analysis of Numerical features', fontsize=20, fontweight='bold', y=1.02)

# for i,col in enumerate(numeric_features[1:]):
#   plt.subplot(2, 4, i+1)                     # subplots of 2 rows and 4 columns

#   # regression plots
#   sns.regplot(x=risk_df[col], y=risk_df['TenYearCHD'])
#   # x-axis lable
#   plt.xlabel(col)
#   plt.tight_layout()

### Counting number of category present in each feature with respect to target feature  

In [ ]:
plt.figure(figsize=(15,5))
# title
plt.suptitle('Bivariate Analysis of Categorical Features', fontsize=20, fontweight='bold', y=1.02)

for i,col in enumerate(categorical_features[:-1]):       # taking all features in categoric column except target feature(TenYearCHD) 
  plt.subplot(2, 4, i+1)                                 # subplots of 2 rows and 4 columns
  a = risk_df.groupby(col)[['TenYearCHD']].mean().reset_index()

  # barplot
  sns.barplot(x=a[col], y=a['TenYearCHD'])
  # x-axis label
  plt.xlabel(col)
plt.tight_layout()

<span style="color:orange; font-size:24px"> Insight: although the proportion of people with prevalentStroke,diabetes,BPMeds is low but those who have this medical condition are at high risk for chd this imbalance needs to be passed with care to models means proper hyperparameter tuning required eg class_weight='balanced for SVM, setting min_leaf do decision trees</span> 

### Deal with missing values

In [ ]:
# features which has less than 5%  null values present. 

nan_columns = ['education', 'cigsPerDay', 'BPMeds', 'totChol', 'BMI', 'heartRate']
risk_df.dropna(subset=nan_columns, inplace=True)

In [ ]:
risk_df.shape

## using KNN Imputer for glucose values
### glucose level are continuous in nature 

In [ ]:
from sklearn.impute import KNNImputer
knn=KNNImputer(weights='distance')
risk_df['glucose']=knn.fit_transform(risk_df[['glucose']])
risk_df['glucose'].isnull().sum()

### plot mean, median line to check skewness if mean = median then normal if mean > median then positively skewed if mean < median then negatively skewed

In [ ]:
# figsize
plt.figure(figsize=(12,4))
# title
plt.suptitle('Data distibution in numerical columns', fontsize=20, fontweight='bold', y=1.02)

for i,col in enumerate(numeric_features):
  plt.subplot(2, 4, i+1)                      # subplots of 2 rows and 4 columns

  # distplot
  sns.histplot(risk_df[col]) 
  # mean line  
  plt.axvline(risk_df[col].mean(), color='blue', linestyle='dashed', linewidth=2)
  # median line
  plt.axvline(risk_df[col].median(), color='red', linestyle='dashed', linewidth=2)   
  # x-axis label
  plt.xlabel(col)
plt.tight_layout()

## Handling outliers
### using capping method

In [ ]:
def clip_outliers(risk_df):
    for col in risk_df[numeric_features]:
        # using IQR method to define range of upper and lower limit.
        q1 = risk_df[col].quantile(0.25)
        q3 = risk_df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        # replacing the outliers with upper and lower bound
        risk_df[col] = risk_df[col].clip(lower_bound, upper_bound)
    return risk_df

In [ ]:
# using the function to treat outliers
risk_df = clip_outliers(risk_df)

 <span style="color:orange"> checking the boxplot after outlier treatment </span>

In [ ]:
plt.figure(figsize=(12,4))

plt.suptitle('Data Distibution of Numeric Features', fontsize=20, fontweight='bold', y=1.02)

for i,col in enumerate(numeric_features):

  plt.subplot(2, 4, i+1)                       # subplots 2 rows, 4 columns

  # kde plots
  sns.boxplot(y=risk_df[col])  
  # x-axis label
  plt.xlabel(col)
  
plt.tight_layout()

In [ ]:
# checking for distribution after treating outliers.
plt.figure(figsize=(12,4))

plt.suptitle('Data Distibution of Numeric Features', fontsize=20, fontweight='bold', y=1.02)

for i,col in enumerate(numeric_features):
  plt.subplot(2, 4, i+1)                       # subplots 2 rows, 4 columns
  # kde plots
  sns.kdeplot(risk_df[col])  
  # x-axis label
  plt.xlabel(col)
plt.tight_layout()

In [ ]:
risk_df['sex'] = risk_df['sex'].map({'M':1, 'F':0})
risk_df['is_smoking'] = risk_df['is_smoking'].map({'YES':1, 'NO':0})

In [ ]:
# Check Unique Values for each categorical variable.
for i in categorical_features:
  print("No. of unique values in",i,"is",risk_df[i].nunique())

In [ ]:
# dropping our target variable from categorical features list
categorical_features.remove('TenYearCHD')

In [ ]:
# one-hot encode the 'education' feature
dummies = pd.get_dummies(risk_df['education'])
dummies.columns = [
    'education_1',
    'education_2',
    'education_3',
    'education_4'
]

# concatenate the one-hot encoded education feature with the rest of the data
risk_df = pd.concat([risk_df, dummies], axis=1)
# drop the original education feature
risk_df=risk_df.drop(['education','education_4'], axis=1,errors='ignore')
risk_df.head(3)

In [ ]:
# Plotting correlation heatmap
plt.figure(figsize=(12,4))
sns.heatmap(risk_df.corr(),annot=True)

Multicollinearity happens when features contain almost the same information, making weight-based models unable to assign stable and meaningful coefficients.

Highly correlated features make it impossible for weight-based models to uniquely and stably determine the coefficients.

This ain't an issue with split based models.

In [ ]:
'''find and remove correlated features'''

def correlation(dataset, threshold):
    corr_matrix = dataset.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) > threshold:        # we are interested in absolute coeff value
                print(
                    corr_matrix.columns[i],",",
                    corr_matrix.columns[j],
                )


In [ ]:
# checking the highly correlated features
correlation(risk_df, 0.7)

In [ ]:
# checking data, weather the provide information is correct or not
risk_df[(risk_df.is_smoking == 'YES') & (risk_df.cigsPerDay == 0)] # it will return those rows where is_smoking is yes but cigsPerDay is 0

Insight: Hence not inconsistent data and is_smoking information can be derived from cigsPerDay hence drop is_smoking feature

In [ ]:
risk_df=risk_df.drop(['is_smoking'],axis=1,errors='ignore')

In [ ]:
risk_df['sysBP'].corr(risk_df['TenYearCHD'])

In [ ]:
risk_df['diaBP'].corr(risk_df['TenYearCHD'])

hence reomove diaBP

In [ ]:
risk_df=risk_df.drop(['diaBP','prevalentHyp'],axis=1,errors='ignore')
risk_df.sample(2)